In [1]:
import pandas as pd
import numpy as np
import scipy.io as scio
import mat73
import os
import sys
from tqdm import tqdm
from collections import defaultdict
import openpyxl 
import dill
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import plotly.graph_objs as go
from matplotlib.animation import FuncAnimation
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.model_selection import train_test_split
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import permutation_test_score
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.cross_decomposition import PLSRegression
from sklearn.preprocessing import RobustScaler
from statsmodels.stats.multitest import multipletests
from sklearn.model_selection import StratifiedGroupKFold #try this or RepeatedStratifiedKFold
import importlib
import helper_function as hf
importlib.reload(hf)
from gtda.time_series import TakensEmbedding

2026-04-16 16:31:45,269 [INFO] 
Limited Total Variation Regularization Support Detected! 
---> CVXPY is not installed. 
---> Many Total Variation Methods require CVXPY including: 
---> velocity, acceleration, jerk, jerk_sliding, smooth_acceleration
---> Please install CVXPY to use these methods.
---> Recommended to also install MOSEK and obtain a MOSEK license.
You can still use: total_variation_regularization.iterative_velocity

2026-04-16 16:31:45,277 [INFO] 
Limited Linear Model Support Detected! 
---> PYCHEBFUN is not installed. 
---> Install pychebfun to use chebfun derivatives (https://github.com/pychebfun/pychebfun/) 
You can still use other methods 

2026-04-16 16:31:45,278 [INFO] 
Limited Linear Model Support Detected! 
---> CVXPY is not installed. 
---> Install CVXPY to use lineardiff derivatives 
You can still use other methods 



In [2]:
#IMPORT data from matlab files
dd= pd.read_csv("/lisc/data/scratch/neurobiology/zimmer/jalaja/imaging/newline/cleaned/for_BundleNet/deltaFoF_alldataset.csv", delimiter=',',parse_dates=True)
dd.columns=dd.columns.str.strip("''")
dd

,AIBL,AIBR,AIZL,ALA,ALNL,ALNR,AQR,AS10,ASKL,ASKR,...,URYVR,VA01,VA11,VA12,VB01,VB02,VD13,target,state,group
0,0.387170,0.097740,0.312340,0.145770,NaN,0.95588,-0.11433,0.003151,1.35830,0.554600,...,0.073543,0.12440,NaN,NaN,0.62955,0.098488,0.015532,2,1,1
1,0.418530,0.069665,0.267290,0.128760,NaN,0.92111,-0.13575,-0.015684,1.27280,0.570670,...,0.046784,0.12474,NaN,NaN,0.61028,0.084541,0.013973,2,1,1
2,0.448780,0.051929,0.239980,0.105750,NaN,0.87986,-0.14115,-0.037594,1.16260,0.580160,...,0.033031,0.12504,NaN,NaN,0.57180,0.079309,-0.002591,2,1,1
3,0.477770,0.045791,0.232590,0.076009,NaN,0.83133,-0.12856,-0.062951,1.02460,0.582280,...,0.033870,0.12532,NaN,NaN,0.51179,0.083855,-0.035988,2,1,1
4,0.475250,0.053608,0.209490,0.073328,NaN,0.85503,-0.13928,-0.052906,0.95813,0.536360,...,0.039577,0.13207,NaN,NaN,0.51291,0.082970,-0.007096,2,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
85045,-0.100090,-0.123270,-0.091191,-0.339340,-0.35703,-0.48824,-0.22803,NaN,-0.19723,-0.132620,...,-0.126310,NaN,-0.052055,0.103360,-0.18571,-0.098487,0.198120,2,1,9
85046,-0.086111,-0.106290,-0.089964,-0.339710,-0.34738,-0.48779,-0.20283,NaN,-0.19967,-0.128080,...,-0.127810,NaN,-0.058894,0.097431,-0.19207,-0.100030,0.206260,2,1,9
85047,-0.089498,-0.134590,-0.097088,-0.355450,-0.33665,-0.49817,-0.20000,NaN,-0.19820,-0.117480,...,-0.114520,NaN,-0.066565,0.095080,-0.20473,-0.103260,0.214000,2,1,9
85048,-0.090512,-0.149300,-0.092970,-0.346930,-0.33888,-0.49830,-0.20349,NaN,-0.20218,-0.108090,...,-0.120090,NaN,-0.053817,0.091680,-0.20745,-0.111210,0.203250,2,1,9


In [3]:
#Normalize scaling beaseline and stimulus period separately because of special request
fps = 5
# simple mean imputation from sklearn
imp=SimpleImputer(strategy='mean')
X=dd.drop({'target','state','group'},axis=1)
#Normalize scaling
scaled_datasets_X_baseline = []
scaled_datasets_X_stimulus = []
for n in range(0, 9):
    #robust_scaler=StandardScaler()
    robust_scaler=RobustScaler(with_centering=False, with_scaling=True, quantile_range=(5, 95))
    scaled_data_X_baseline = robust_scaler.fit_transform(X.loc[n*9450:(n*9450)+(450*fps)-1,:])
    scaled_data_X_stimulus = robust_scaler.fit_transform(X.loc[(n*9450)+(450*fps):((n+1)*9450)-1,:])
    
    scaled_datasets_X_baseline.append(scaled_data_X_baseline)
    scaled_datasets_X_stimulus.append(scaled_data_X_stimulus)

# Concatenate scaled datasets from each group
scaled_X_within_group_baseline = np.concatenate(scaled_datasets_X_baseline)
scaled_X_within_group_stimulus = np.concatenate(scaled_datasets_X_stimulus)

# Scale across all groups
scaled_X_across_groups_baseline = robust_scaler.fit_transform(scaled_X_within_group_baseline)
scaled_X_baseline =pd.DataFrame(data=scaled_X_across_groups_baseline,columns=X.columns)

scaled_X_across_groups_stimulus = robust_scaler.fit_transform(scaled_X_within_group_stimulus)
scaled_X_stimulus =pd.DataFrame(data=scaled_X_across_groups_stimulus,columns=X.columns)

normalized=[]
for n in range(0, 9):
   
    normalized.append(scaled_X_baseline.loc[n*450*fps:((n+1)*450*fps)-1,:])
    normalized.append(scaled_X_stimulus.loc[n*1440*fps:((n+1)*1440*fps)-1,:])
    
# Combine all normalized data
normalized = pd.concat(normalized, axis=0).reset_index(drop=True)

# Add back the target, state, and group columns
normalized_data = normalized.copy()
normalized_data['target'] = dd['target'].values
normalized_data['state'] = dd['state'].values
normalized_data['group'] = dd['group'].values

# Ensure column names are clean
normalized_data.columns = normalized_data.columns.str.strip("''")

/lisc/data/scratch/neurobiology/zimmer/jalaja/conda/envs/jupyter/lib/python3.8/site-packages/numpy/lib/nanfunctions.py:1556: RuntimeWarning: All-NaN slice encountered
  return function_base._ureduce(a,
/lisc/data/scratch/neurobiology/zimmer/jalaja/conda/envs/jupyter/lib/python3.8/site-packages/numpy/lib/nanfunctions.py:1556: RuntimeWarning: All-NaN slice encountered
  return function_base._ureduce(a,
/lisc/data/scratch/neurobiology/zimmer/jalaja/conda/envs/jupyter/lib/python3.8/site-packages/numpy/lib/nanfunctions.py:1556: RuntimeWarning: All-NaN slice encountered
  return function_base._ureduce(a,
/lisc/data/scratch/neurobiology/zimmer/jalaja/conda/envs/jupyter/lib/python3.8/site-packages/numpy/lib/nanfunctions.py:1556: RuntimeWarning: All-NaN slice encountered
  return function_base._ureduce(a,
/lisc/data/scratch/neurobiology/zimmer/jalaja/conda/envs/jupyter/lib/python3.8/site-packages/numpy/lib/nanfunctions.py:1556: RuntimeWarning: All-NaN slice encountered
  return function_base._u

In [ ]:
dk=hf.impute_missing_values_in_dataframe(normalized_data.drop({'target','state','group'}, axis=1))
df = pd.DataFrame(data=np.c_[dk, dd['target'],dd['state'],dd['group']],columns=dd.columns)

/lisc/data/scratch/neurobiology/zimmer/jalaja/conda/envs/jupyter/lib/python3.8/site-packages/ppca/_ppca.py:82: RuntimeWarning: divide by zero encountered in log
  det = np.log(np.linalg.det(Sx))


In [ ]:
df = hf.fill_short_states(df, 'state', max_length=3)
print(df)

In [ ]:
annotations = df["state"]
conditions = [
    ((annotations) == 1.0),
    ((annotations)  == 2.0),
    ((annotations)  == 3.0),
    ((annotations)  == 4.0)
]

values = ['forward', 'reversal', 'sustained reversal', 'turn']

# replace values based on conditions, 1 = forward, 2 = reversal (rise), 3 = reversal (sustained), 4 = turn
transformed_annotations = np.select(conditions, values)

df["state"] = hf.determine_turn(df, transformed_annotations) # to determine whether a turn is a dorsal or a ventral turn


In [ ]:
state_mapping = {
    'forward': 1,
    'reversal': 2,
    'sustained reversal': 3,
    'ventral': 4,
    'dorsal': 5  # Adjust based on your specific categories
}

# Apply the mapping to the 'state' column
df['state'] = df['state'].replace(state_mapping)

# Check the results
print(df['state'].unique())

In [ ]:
# Find isolated runs of target == 1
isolated_indices = []

for i in range(1, len(df) - 1):
    if df['target'].iloc[i] == 1 and df['target'].iloc[i-1] != 1 and df['target'].iloc[i+1] != 1:
        isolated_indices.append(i)

# Replace the isolated instances of target == 1 with 2
df.loc[isolated_indices, 'target'] = 2

In [ ]:
# remove sensory neurons for now!
df_no_sensory=df.drop({'URXL','URXR','AQR','PQR','AUAL','AUAR','RMGL','RMGR','PVPL','IL2L','IL2R','BAGL','BAGR'},axis=1)
df_sensory=df[['URXL','URXR','AQR','PQR','AUAL','AUAR','RMGL','RMGR','PVPL','IL2L','IL2R','BAGL','BAGR','target','state','group']]

In [ ]:
### decision, used these for the plot###
pls = PLSRegression(n_components=5)
cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=1, random_state=None)
lda = LinearDiscriminantAnalysis()


In [ ]:
interval_start = 450 # start at 390 seconds
interval_step = 90  # interval of 90 seconds
slice_duration = 8.5 # slice the prestimulus 60 seconds
stimulus_period=30
fps = 5  # frames per second

# Create the new sliced DataFrame
prestim_sliced_df , prestim_triggered_df= hf.create_sliced_dataframe(df, interval_start, interval_step, slice_duration,stimulus_period, fps)
print(triggered_df)

In [1]:
df_pls=pd.DataFrame(data=prestim_sliced_df)
# Drop rows where 'target_vector' is 2
sliced_df_forPLS = df_pls[df_pls['target'] != 2]
pls_x=sliced_df_forPLS.drop({'target','state','group'},axis=1)

pls_y=sliced_df_forPLS['target']
pls_groups=sliced_df_forPLS['group']

NameError: name 'pd' is not defined

In [ ]:
pls.fit(pls_x,pls_y)
pls_df_for_lda_stim=pd.DataFrame(data=pls.transform(pls_x), columns=['PLS1','PLS2','PLS3','PLS4','PLS5'])
cv_scores = cross_val_score(lda,pls_df_for_lda_stim, pls_y, scoring='accuracy', cv=cv, groups=pls_groups, n_jobs=-1)
cv_scores.mean()

In [ ]:
bin_duration = 8.5  # Bin duration in seconds
bin_frames = int(bin_duration * fps)  # Number of frames per bin
# Function to process data for runs and bins
def find_runs(data):
    runs = []
    start_indices = []
    start_idx = None
    
    for idx, value in enumerate(data):
        if value == 2:
            if start_idx is None:
                start_idx = idx
        else:
            if start_idx is not None:
                run_length = idx - start_idx
                if run_length > 3:
                    runs.append((start_idx, run_length))
                    start_indices.append(start_idx)
                start_idx = None
    
    # Check for a run that goes to the end of the data
    if start_idx is not None:
        run_length = len(data) - start_idx
        if run_length > 3:
            runs.append((start_idx, run_length))
            start_indices.append(start_idx)
    
    return runs, start_indices

def process_data(data, group, stimulus_data_11,aug_step):
    run_data_list = []
    valid_indices = set(stimulus_data_11.index)  # Get valid index range from stimulus_data_11

    # Extract bins for the identified runs (target = 1)
    runs, start_indices = find_runs(data['state'].values)
    for start_idx in start_indices:
        bin_start = start_idx - bin_frames
        bin_end = start_idx
        if bin_start >= 0 and bin_end < len(data):
            bin_data = data.iloc[bin_start:bin_end]
            # Ensure all rows in bin_data belong to stimulus_data_11
            if (
                bin_data['state'].isin([1, 4, 5]).all() and
                set(bin_data.index).issubset(valid_indices)
            ):
                bin_entry = bin_data.copy()
                bin_entry['target'] = 1
                bin_entry['start_index'] = start_idx
                bin_entry['group'] = group
                run_data_list.append(bin_entry)
                
                augmented_bins = augment_bin(bin_data, bin_start, bin_end, data, group, start_idx,aug_step,stimulus_data_11)
                
                # Add the augmented bins to the group
                run_data_list.extend(augmented_bins)

    # Find non-overlapping bins (target = 0)
    i = 0
    while i <= len(data) - bin_frames:
        bin_data = data.iloc[i-bin_frames:i]
        # Ensure all rows in bin_data belong to stimulus_data_11 and it's non-overlapping
        if (
            bin_data['state'].isin([1, 4, 5]).all() and
            set(bin_data.index).issubset(valid_indices) and
            all((i < start_idx - bin_frames-aug_step or i >= start_idx) for start_idx in start_indices)
        ):
            bin_entry = bin_data.copy()
            bin_entry['target'] = 0
            bin_entry['start_index'] = i
            bin_entry['group'] = group
            run_data_list.append(bin_entry)
            i += bin_frames  # Advance by bin_frames to avoid overlap
        else:
            i += 1  # Increment by 1 to check the next bin

    return run_data_list
    
def augment_bin(bin_data, bin_start, bin_end, data, group, start_idx, aug_step, stimulus_data_11):
    augmented_bins = []
    valid_indices = set(stimulus_data_11.index)  # Valid indices from stimulus_data_11
    
    for shift in range(0, aug_step):  # Shift up to `aug_step` frames back
        # Shift the start and end of the bin by 'shift' frames backward
        shifted_bin_start = bin_start - shift
        shifted_bin_end = bin_end - shift
        
        # Ensure the shifted bin is within the bounds of the data
        if shifted_bin_start >= 0:
            shifted_bin_data = data.iloc[shifted_bin_start:shifted_bin_end]
            
            # Check if the shifted bin contains only valid states (1 or 4) and belongs to stimulus_data_11
            if (
                shifted_bin_data['state'].isin([1, 4, 5]).all() and
                set(shifted_bin_data.index).issubset(valid_indices)
            ):
                augmented_bin = shifted_bin_data.copy()
                augmented_bin['target'] = 1
                augmented_bin['start_index'] = start_idx  # Keep the original start index
                augmented_bin['group'] = group  # Track the group
                augmented_bins.append(augmented_bin)
    
    return augmented_bins


In [ ]:
# Initialize lists for all categories
run_data_list = []
# Loop through each group in the data
for group in df['group'].unique():
    # Filter data for the current group
    group_data = df[df['group'] == group]
    baseline_data = group_data.iloc[:390 * fps, :]
    stimulus_data = group_data.iloc[390 * fps:, :]
    # Define response_data
    response_frames = [
        (int(start * fps), int(end * fps))
        for start, end in zip(start_frames, end_frames)
    ]
    response_data = pd.concat([
        group_data.iloc[start:end] for start, end in response_frames if end <= len(group_data)
    ], ignore_index=True)
    
    # Define stimulus_data_21
    stimulus_21_frames = [(int(end * fps), int((start + 30) * fps)) for end, start in zip(end_frames,start_frames)]
    
    stimulus_data_21 = pd.concat([
    group_data.iloc[start:end]
    for start, end in stimulus_21_frames
    if end <= len(group_data)], ignore_index=True)
    
    # Define stimulus_data_11
    stimulus_11_frames = [(int((start+30) * fps), int((start + 90) * fps)) for start in start_frames]
    stimulus_data_11 = pd.concat([
        group_data.iloc[start:end]
        for start, end in stimulus_11_frames])
    
    # Process baseline_data
    run_data_list.extend(process_data(group_data, group,stimulus_data_11,aug_step=8))
    # Process whole_data
    # run_data_list.extend(process_data(group_data, group,group_data,aug_step=5))
# Combine all data into DataFrames
run_dataframe = pd.concat(run_data_list,ignore_index=True)
print(run_dataframe)

In [ ]:
X= run_dataframe.drop(['group', 'target', 'state','start_index'], axis=1)
y= run_dataframe['target']
pls_df_for_lda=pd.DataFrame(data=pls.transform(X), columns=['PLS1','PLS2','PLS3','PLS4','PLS5'])
groups=run_dataframe['group']
cv_scores = cross_val_score(lda,pls_df_for_lda, y, scoring='accuracy', cv=cv, groups=groups, n_jobs=-1)
cv_scores.mean()
# cv_scores.std()

In [ ]:
# Initialize lists for all categories
run_data_list = []
# Loop through each group in the data
for group in df['group'].unique():
    # Filter data for the current group
    group_data = df_no_sensory[df['group'] == group]
    baseline_data = group_data.iloc[:390 * fps, :]
    stimulus_data = group_data.iloc[390 * fps:, :]
    # Define response_data
    response_frames = [
        (int(start * fps), int(end * fps))
        for start, end in zip(start_frames, end_frames)
    ]
    response_data = pd.concat([
        group_data.iloc[start:end] for start, end in response_frames if end <= len(group_data)
    ], ignore_index=True)
    
    # Define stimulus_data_21
    stimulus_21_frames = [(int(end * fps), int((start + 30) * fps)) for end, start in zip(end_frames,start_frames)]
    
    stimulus_data_21 = pd.concat([
    group_data.iloc[start:end]
    for start, end in stimulus_21_frames
    if end <= len(group_data)], ignore_index=True)
    
    # Define stimulus_data_11
    stimulus_11_frames = [(int((start+30) * fps), int((start + 90) * fps)) for start in start_frames]
    stimulus_data_11 = pd.concat([
        group_data.iloc[start:end]
        for start, end in stimulus_11_frames])
    
    # Process baseline_data
    run_data_list.extend(process_data(group_data, group,stimulus_data_11,aug_step=8))
    # Process whole_data
    # run_data_list.extend(process_data(group_data, group,group_data,aug_step=5))
# Combine all data into DataFrames
run_dataframe = pd.concat(run_data_list,ignore_index=True)
print(run_dataframe)

In [ ]:
X= run_dataframe.drop(['group', 'target', 'state','start_index'], axis=1)
y= run_dataframe['target']
pls = PLSRegression(n_components=5)  # You can adjust the number of components
# Initialize LDA model
pls.fit(X, y)
pls_df_for_lda=pd.DataFrame(data=pls.transform(X), columns=['PLS1','PLS2','PLS3','PLS4','PLS5'])
groups=run_dataframe['group']

cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=1, random_state=None)
cv_scores = cross_val_score(lda,pls_df_for_lda, y, scoring='accuracy', cv=cv, groups=groups, n_jobs=-1)
cv_scores.mean()
# cv_scores.std()

In [ ]:
pls_df_for_lda_stim=pd.DataFrame(data=pls.transform(pls_x), columns=['PLS1','PLS2','PLS3','PLS4','PLS5'])
cv_scores = cross_val_score(lda,pls_df_for_lda_stim, pls_y, scoring='accuracy', cv=cv, groups=pls_groups, n_jobs=-1)
cv_scores.mean()
# cv_scores.std()

In [ ]:
#permutation test of sensory PLS model
from sklearn.model_selection import permutation_test_score
from numpy import mean
scores, perm_scores, pvalue = permutation_test_score(lda, pls_df_for_lda_stim, pls_y, scoring="accuracy", cv=cv, groups=pls_groups,n_permutations=2000)
print('Mean Accuracy: %.3f (%.3f)' % (mean(scores), mean(perm_scores)))
pvalue

In [ ]:
# permutation test of spontaneous PLS model
scores, perm_scores, pvalue = permutation_test_score(lda, pls_df_for_lda, y, scoring="accuracy", cv=cv, groups=groups,n_permutations=2000)
print('Mean Accuracy: %.3f (%.3f)' % (mean(scores), mean(perm_scores)))
pvalue

In [ ]:
#simply used the above values

In [ ]:
# Define your confusion matrix data normalized separately, 8.5 sec window and 5 component
confusion_matrix = np.array([[71.8, 63.4],
                             [60.7, 81.9]])

# Define the range for the colorbar
vmin = 50  # Minimum value for the colorbar
vmax = 100  # Maximum value for the colorbar

# Plot the heatmap with custom colorbar range
plt.figure(figsize=(10, 8))
ax=sns.heatmap(
    confusion_matrix,
    annot=True,
    fmt=".1f",
    cmap="cividis",  # Diverging colormap for accuracy
    cbar=True,
    cbar_kws={"label": "Accuracy (%)" },  # Label for the colorbar
    vmin=vmin,
    vmax=vmax,
    annot_kws={"size": 20}  # Font size for annotations
)

# Customize colorbar font size
cbar = ax.collections[0].colorbar
cbar.ax.tick_params(labelsize=16)  # Font size for tick labels
cbar.ax.set_ylabel("Accuracy (%)", fontsize=20)  # Font size for the label
# Add labels, title, and ticks with larger font size
plt.xlabel("Models", fontsize=20)
plt.ylabel("Data", fontsize=20)
plt.title("Confusion Matrix with Custom Colorbar Range", fontsize=18)
plt.xticks(ticks=np.arange(confusion_matrix.shape[1]) + 0.5,
           labels=["Spontaneous PLS model", "Sensory PLS model"],
           fontsize=20)
plt.yticks(ticks=np.arange(confusion_matrix.shape[0]) + 0.5,
           labels=["Spontaneous data", "Sensory data"],
           rotation=90,
           fontsize=20)

# Adjust layout for better fit
plt.tight_layout()
plt.rcParams['svg.fonttype'] = 'none'
# plt.savefig("pls_models_accuracy_matrix.svg", format='svg',dpi=300)
plt.show()